# Red Neuronal para Predicción de Impago de Tarjetas de Crédito

Este notebook contiene el código paso a paso para construir, entrenar y evaluar una red neuronal utilizando el dataset `default of credit card clients.csv`. Está diseñado con celdas de Markdown (texto) que explican cada paso del proceso, siguiendo un enfoque metodológico de Ciencia de Datos.

## 1. Importación de Librerías

Para este proyecto utilizaremos:
* **Pandas y Numpy:** Para la manipulación y análisis de datos.
* **Scikit-Learn (sklearn):** Para dividir los datos, escalarlos y evaluar el modelo (matriz de confusión).
* **TensorFlow / Keras:** Para construir la arquitectura de la red neuronal profunda.
* **Matplotlib y Seaborn:** Para visualizar los resultados (mapas de calor).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Ignorar advertencias para una salida más limpia
import warnings
warnings.filterwarnings('ignore')

## 2. Carga y Exploración de Datos

Cargamos el archivo CSV. Basándonos en la estructura del archivo, este utiliza un punto y coma (`;`) como separador y tiene un doble encabezado (las variables X1, X2 y luego los nombres descriptivos). Nos quedaremos con los nombres descriptivos. Además, eliminamos la columna `ID` ya que es un identificador único que no tiene poder predictivo y podría confundir a la red neuronal.

In [ ]:
# Cargar el dataset saltando la primera fila genérica (X1, X2, etc.)
try:
    df = pd.read_csv('default of credit card clients.csv', sep=';', skiprows=1)
except FileNotFoundError:
    print("Por favor, asegúrate de que el archivo 'default of credit card clients.csv' esté en el mismo directorio.")

# Renombrar la variable objetivo a algo más sencillo
if 'default payment next month' in df.columns:
    df.rename(columns={'default payment next month': 'default'}, inplace=True)

# Eliminar la columna ID
if 'ID' in df.columns:
    df.drop('ID', axis=1, inplace=True)

# Mostrar las primeras 5 filas
display(df.head())

## 3. Preprocesamiento de Datos (Crítico)

Las redes neuronales son extremadamente sensibles a la escala de los datos. Si tenemos una columna con valores en los cientos de miles (como el límite de crédito `LIMIT_BAL`) y otra con valores pequeños (como el estado de pago `PAY_0` que va de -2 a 8), los gradientes durante el entrenamiento pueden volverse inestables.

* **Separación X e y:** Dividimos en variables predictoras (`X`) y variable objetivo (`y`).
* **División Train/Test:** 80% de los datos para entrenar la red, 20% para probar que no se haya memorizado los datos (overfitting).
* **Escalado:** Usamos `StandardScaler` para que todas las variables tengan media 0 y desviación estándar 1.

In [ ]:
# 1. Separar variables predictoras (X) y objetivo (y)
X = df.drop('default', axis=1)
y = df['default']

# 2. Dividir en conjuntos de entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Escalar los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamaño del set de entrenamiento: {X_train_scaled.shape}")
print(f"Tamaño del set de prueba: {X_test_scaled.shape}")

## 4. Construcción de la Red Neuronal

Definimos una arquitectura *Feedforward* (Secuencial).
* **Capa de entrada:** Automáticamente inferida por la primera capa oculta (espera 23 características).
* **Capas ocultas:** Dos capas densas (`Dense`) con activación `ReLU` para capturar patrones no lineales complejos.
* **Capa de salida:** Una sola neurona con activación `sigmoid`. Esta función comprime el resultado entre 0 y 1, dándonos una "probabilidad" de que el cliente incumpla (1).

In [ ]:
# Inicializar el modelo secuencial
model = Sequential()

# Primera capa oculta (y capa de entrada implícita)
model.add(Dense(units=32, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dropout(0.2)) # Apaga el 20% de las neuronas al azar para evitar sobreajuste

# Segunda capa oculta
model.add(Dense(units=16, activation='relu'))

# Capa de salida
# Una neurona, activación sigmoide para clasificación binaria (0 o 1)
model.add(Dense(units=1, activation='sigmoid'))

# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar el resumen de la arquitectura
model.summary()

## 5. Entrenamiento del Modelo

Entrenamos la red pasando los datos varias veces (`epochs`). En cada época (iteración), el modelo ajusta sus "pesos" internos para reducir el error (`loss`). Usamos un `batch_size` de 32, lo que significa que actualiza los pesos cada 32 ejemplos.

In [ ]:
# Entrenar la red neuronal
history = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=20,          # Número de veces que verá todos los datos
    batch_size=32,      # Muestras por actualización de gradiente
    validation_split=0.2, # Usar 20% del set de train para validación interna
    verbose=1
)

## 6. Evaluación y Visualización (Resultados)

Como sugerimos para tu exposición, la métrica global de precisión (`accuracy`) no es suficiente en problemas financieros. Generamos las predicciones y visualizamos la **Matriz de Confusión** utilizando un mapa de calor (`heatmap`). Esto permite ver:
* **Verdaderos Positivos / Negativos:** Aciertos del modelo.
* **Falsos Negativos:** Clientes que iban a incumplir, pero el modelo confió en ellos (el error más costoso para el banco).
* **Falsos Positivos:** Clientes cumplidos a los que el modelo marcó como riesgo.

In [ ]:
# Obtener predicciones (probabilidades)
y_pred_prob = model.predict(X_test_scaled)

# Convertir probabilidades a clases binarias (0 o 1) usando un umbral de 0.5
y_pred = (y_pred_prob > 0.5).astype(int)

# --- MATRIZ DE CONFUSIÓN ---
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Cumple (0)', 'Incumple (1)'], 
            yticklabels=['Cumple (0)', 'Incumple (1)'])
plt.title('Matriz de Confusión - Red Neuronal')
plt.ylabel('Valor Real')
plt.xlabel('Predicción del Modelo')
plt.show()

# --- REPORTE DE CLASIFICACIÓN ---
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['Cumple (0)', 'Incumple (1)']))